# 06 - 複雜多模型 Workflow：Markdown 表格 + 圖片分析 + 評估

本筆記本展示一個完整的複雜案例：

1. **多模型配置**：使用新版 `llm_config.yaml` 的 `shared_config` + `model_configs`
2. **Workflow 建構**：LangGraph workflow 中存入 `llm_config` + `prompt_template`
3. **多模型切換**：在不同 node 中使用 `set_model` 切換模型
4. **Prompt 組裝**：使用 `mlflow.genai` 載入 prompt，將 markdown table 帶入 prompt
5. **圖片輸入**：支援 base64 圖片
6. **評估**：使用 MLflow GenAI evaluate 進行完整評估

## Step 1: 環境初始化

In [ ]:
import sys
import os
import base64

sys.path.insert(0, os.path.abspath(".."))

import mlflow
from app.utils.config import init_config
from app.logger import init_mlflow

cfg = init_config()
init_mlflow(cfg)

print("環境初始化完成。")

## Step 2: 載入多模型配置

新版 `llm_config.yaml` 結構：
- `shared_config`：共用設定（J1→J2 auth URL 分 DEV/TEST/STG/PROD）
- `model_configs`：各模型獨立設定（QWEN3、QWEN3VL 等）

In [ ]:
from llm_service import LLMConfig, SharedConfig, ModelConfig

# 方法 1：從 llm_config.yaml 載入
# cfg = LLMConfig.from_yaml()

# 方法 2：程式中直接建構（Demo 用）
llm_cfg = LLMConfig(
    shared_config=SharedConfig(
        default_zone="DEV",
        auth_urls={
            "DEV": "https://auth-dev.internal.com/api/token",
            "TEST": "https://auth-test.internal.com/api/token",
            "STG": "https://auth-stg.internal.com/api/token",
            "PROD": "https://auth-prod.internal.com/api/token",
        },
    ),
    model_configs={
        "QWEN3": ModelConfig(
            j1_token="demo-j1-token",
            model_name="qwen3",
            api_endpoints={
                "DEV": "https://llm-dev.internal.com/v1",
                "TEST": "https://llm-test.internal.com/v1",
                "STG": "https://llm-stg.internal.com/v1",
                "PROD": "https://llm-prod.internal.com/v1",
            },
            max_tokens=4096,
            temperature=0.7,
        ),
        "QWEN3VL": ModelConfig(
            j1_token="demo-j1-token-vl",
            model_name="qwen3-vl",
            api_endpoints={
                "DEV": "https://llm-vl-dev.internal.com/v1",
                "TEST": "https://llm-vl-test.internal.com/v1",
                "STG": "https://llm-vl-stg.internal.com/v1",
                "PROD": "https://llm-vl-prod.internal.com/v1",
            },
            max_tokens=4096,
            temperature=0.3,
        ),
    },
)

print(f"可用模型: {llm_cfg.list_models()}")
print(f"預設 Zone: {llm_cfg.shared_config.default_zone}")

## Step 3: 準備資料 — Markdown 表格 + 圖片

模擬真實場景：
- 兩個 markdown table（銷售數據 + 產品資訊）
- 一張 base64 編碼的圖片（此處用簡單的 1x1 pixel PNG 模擬）

In [ ]:
# === Markdown 表格 1：銷售數據 ===
sales_table = """| 月份 | 銷售額（萬元）| 成長率 | 客戶數 |
|------|-------------|--------|--------|
| 一月 | 120         | -      | 45     |
| 二月 | 135         | 12.5%  | 52     |
| 三月 | 148         | 9.6%   | 58     |
| 四月 | 162         | 9.5%   | 63     |
| 五月 | 155         | -4.3%  | 59     |
| 六月 | 178         | 14.8%  | 71     |"""

# === Markdown 表格 2：產品資訊 ===
product_table = """| 產品名稱 | 類別   | 單價（元）| 庫存量 | 銷量佔比 |
|---------|--------|----------|--------|--------|
| Alpha   | 硬體   | 2,500    | 320    | 35%    |
| Beta    | 軟體   | 800      | ∞      | 25%    |
| Gamma   | 服務   | 1,200    | -      | 20%    |
| Delta   | 硬體   | 3,800    | 150    | 15%    |
| Epsilon | 配件   | 350      | 500    | 5%     |"""

# === Base64 圖片（1x1 紅色 PNG 模擬圖表）===
# 實際使用時替換為真實的圖表截圖 base64
SAMPLE_IMAGE_BYTES = (
    b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01'
    b'\x00\x00\x00\x01\x08\x02\x00\x00\x00\x90wS\xde\x00'
    b'\x00\x00\x0cIDATx\x9cc\xf8\x0f\x00\x00\x01\x01\x00'
    b'\x05\x18\xd8N\x00\x00\x00\x00IEND\xaeB`\x82'
)
image_base64 = base64.b64encode(SAMPLE_IMAGE_BYTES).decode("utf-8")

print("=== 銷售數據表 ===")
print(sales_table)
print("\n=== 產品資訊表 ===")
print(product_table)
print(f"\n圖片 base64 長度: {len(image_base64)} 字元")

## Step 4: 註冊 Prompt Template

使用 `mlflow.genai` 註冊 prompt，模板中預留 `{{ sales_table }}` 和 `{{ product_table }}` 變數。

In [ ]:
from app.prompts import PromptManager

pm = PromptManager(cfg)

# === 文字分析 Prompt（用 QWEN3 處理）===
analysis_template = """你是一位專業的商業數據分析師。請根據以下兩份報表進行綜合分析。

## 銷售數據
{{ sales_table }}

## 產品資訊
{{ product_table }}

請提供：
1. 銷售趨勢分析（哪些月份表現好/差，原因推測）
2. 產品組合建議（根據銷量佔比和庫存狀況）
3. 下半年策略建議（至少 3 點具體建議）

請使用繁體中文回答，格式清晰，包含數據引用。"""

pm.register(
    "business_analysis",
    analysis_template,
    commit_message="商業數據綜合分析 prompt",
)

# === 圖片分析 Prompt（用 QWEN3VL 處理）===
image_analysis_template = """你是一位視覺分析專家。請分析提供的圖表，並結合以下數據表格進行解讀。

## 參考數據
{{ sales_table }}

請描述圖表中的主要趨勢，並與表格數據進行交叉比對。"""

pm.register(
    "image_analysis",
    image_analysis_template,
    commit_message="圖表視覺分析 prompt",
)

print("Prompt 註冊完成。")

## Step 5: 建構多模型 Workflow

Workflow 流程：

```
preprocess（準備 prompt 變數）
    → assemble_prompt（載入並格式化 prompt）
        → call_llm（呼叫 QWEN3 進行文字分析）
            → END
```

state 中的 `llm_config` 和 `model_alias` 控制使用哪個模型。

In [ ]:
from langgraph.graph import END, StateGraph
from app.workflow.state import WorkflowState
from app.workflow.nodes import (
    create_call_llm_node,
    create_set_model_node,
    create_prompt_assembly_node,
)


# === 自訂前處理 node ===
def preprocess(state: dict) -> dict:
    """前處理：從 metadata 中取出表格資料，設定 prompt 變數。"""
    metadata = state.get("metadata", {})
    return {
        "prompt_variables": {
            "sales_table": metadata.get("sales_table", ""),
            "product_table": metadata.get("product_table", ""),
        },
    }


# === 方法 1：使用 build_multimodel_chain（快速建構）===
from app.workflow.build_workflow import build_multimodel_chain

quick_graph = build_multimodel_chain(
    system_prompt="你是一位專業的商業數據分析師。",
    prompt_name="business_analysis",
    preprocess_fn=preprocess,
)

print("快速建構 workflow 完成。")
print(f"Workflow nodes: {list(quick_graph.get_graph().nodes.keys())}")

In [ ]:
# === 方法 2：手動建構更複雜的 workflow ===
# 流程: preprocess → set_model(QWEN3) → assemble_prompt → call_llm_text
#                                                              ↓
#                                                  set_model(QWEN3VL)
#                                                              ↓
#                                                    call_llm_image → END

graph = StateGraph(WorkflowState)

# Node 1: 前處理
graph.add_node("preprocess", preprocess)

# Node 2: 設定文字模型
graph.add_node("set_text_model", create_set_model_node("QWEN3"))

# Node 3: 組裝 prompt
graph.add_node("assemble_prompt", create_prompt_assembly_node(
    prompt_name="business_analysis",
))

# Node 4: 呼叫文字 LLM
graph.add_node("call_llm_text", create_call_llm_node(
    system_prompt="你是一位專業的商業數據分析師，請用繁體中文回答。",
))

# Node 5: 切換到 VL 模型
graph.add_node("set_vl_model", create_set_model_node("QWEN3VL"))

# Node 6: 呼叫 VL LLM（帶圖片）
graph.add_node("call_llm_image", create_call_llm_node(
    system_prompt="你是一位視覺分析專家，請分析圖片並用繁體中文回答。",
    support_images=True,
))

# 定義 edges
graph.set_entry_point("preprocess")
graph.add_edge("preprocess", "set_text_model")
graph.add_edge("set_text_model", "assemble_prompt")
graph.add_edge("assemble_prompt", "call_llm_text")
graph.add_edge("call_llm_text", "set_vl_model")
graph.add_edge("set_vl_model", "call_llm_image")
graph.add_edge("call_llm_image", END)

complex_graph = graph.compile()

print("複雜 workflow 建構完成。")
print(f"Workflow nodes: {list(complex_graph.get_graph().nodes.keys())}")

## Step 6: 執行 Workflow

在 state 中注入 `llm_config`、`model_alias`、`image_base64` 和 `metadata`。

> **注意**：以下呼叫需要實際的 LLM 服務。若無法連線，會顯示錯誤訊息。

In [ ]:
# === 使用快速建構的 workflow（文字分析）===
try:
    result = quick_graph.invoke({
        "messages": [],
        "llm_config": llm_cfg,
        "model_alias": "QWEN3",
        "metadata": {
            "sales_table": sales_table,
            "product_table": product_table,
        },
    })
    print("=== 文字分析結果 ===")
    print(result["messages"][-1].content)
except Exception as e:
    print(f"[提示] 需要實際 LLM 服務: {e}")
    print("以下使用 mock 結果繼續展示 evaluation 流程。")

In [ ]:
# === 使用複雜 workflow（文字 + 圖片分析）===
try:
    result = complex_graph.invoke({
        "messages": [],
        "llm_config": llm_cfg,
        "model_alias": "QWEN3",
        "image_base64": image_base64,
        "metadata": {
            "sales_table": sales_table,
            "product_table": product_table,
        },
    })
    print("=== 複雜 Workflow 結果 ===")
    for i, msg in enumerate(result["messages"]):
        role = getattr(msg, 'type', 'unknown')
        content = msg.content if hasattr(msg, 'content') else str(msg)
        preview = content[:200] + "..." if len(content) > 200 else content
        print(f"\n[{i}] {role}: {preview}")
except Exception as e:
    print(f"[提示] 需要實際 LLM 服務: {e}")

## Step 7: 使用 mlflow.genai 載入 Prompt 並直接呼叫

展示不透過 workflow，直接用 `mlflow.genai.load_prompt()` 載入 prompt，
將 markdown table 帶入後呼叫 LLM。

In [ ]:
from llm_service import get_litellm_kwargs, build_multimodal_messages
import litellm


def call_with_prompt_and_tables(
    prompt_name: str,
    tables: dict,
    image_b64: str | None = None,
    model_alias: str = "QWEN3",
) -> str:
    """載入 MLflow prompt，帶入 markdown table，呼叫 LLM。"""

    # 1. 載入 prompt
    try:
        prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@latest")
        formatted = prompt.format(**tables)
    except Exception:
        # Fallback: 使用 PromptManager
        formatted = pm.load_and_format(prompt_name, **tables)

    # 2. 建構 messages（含圖片）
    if image_b64:
        messages = build_multimodal_messages(
            user_text=formatted,
            image_base64=image_b64,
            system_prompt="你是一位專業分析師。",
        )
    else:
        messages = [
            {"role": "system", "content": "你是一位專業分析師。"},
            {"role": "user", "content": formatted},
        ]

    # 3. 呼叫 LLM
    kwargs = get_litellm_kwargs(llm_cfg, model_alias=model_alias)
    response = litellm.completion(**kwargs, messages=messages)
    return response.choices[0].message.content or ""


# === 執行 ===
try:
    result = call_with_prompt_and_tables(
        prompt_name="business_analysis",
        tables={"sales_table": sales_table, "product_table": product_table},
        model_alias="QWEN3",
    )
    print("=== 直接呼叫結果 ===")
    print(result)
except Exception as e:
    print(f"[提示] 需要實際 LLM 服務: {e}")

## Step 8: 完整評估流程

使用 MLflow GenAI evaluate 對整個 workflow 進行評估。

評估資料包含：
- 多組 markdown table 輸入
- 預期的關鍵字和品質標準
- Rule-based + LLM Judge 評分

In [ ]:
# === 評估資料 ===
eval_data = [
    {
        "inputs": {
            "sales_table": sales_table,
            "product_table": product_table,
        },
        "expectations": {
            "keywords": "銷售,趨勢,建議,產品,分析",
        },
    },
    {
        "inputs": {
            "sales_table": """| 月份 | 收入 | 支出 |\n|------|------|------|\n| Q1 | 500 | 300 |\n| Q2 | 600 | 350 |""",
            "product_table": """| 項目 | 狀態 |\n|------|------|\n| A | 正常 |\n| B | 異常 |""",
        },
        "expectations": {
            "keywords": "收入,支出,分析",
        },
    },
]

print(f"準備了 {len(eval_data)} 筆評估資料。")

In [ ]:
# === 定義 predict_fn ===
# 實際場景：呼叫 workflow 或 LLM
# Demo：使用 mock 結果

MOCK_RESPONSES = {
    0: (
        "## 銷售趨勢分析\n\n"
        "從銷售數據來看，上半年整體呈上升趨勢。六月銷售額達 178 萬元，為最高點，"
        "成長率 14.8%。五月出現 -4.3% 的下滑，可能與季節性因素有關。\n\n"
        "## 產品組合建議\n\n"
        "Alpha 產品佔銷量 35%，是主力產品，建議維持庫存水位。"
        "Beta 軟體產品無庫存限制，可加大推廣力度。\n\n"
        "## 下半年策略建議\n\n"
        "1. 加強 Alpha 和 Beta 的交叉銷售\n"
        "2. 針對五月的下滑趨勢，提前準備促銷活動\n"
        "3. 增加 Gamma 服務類產品的曝光度"
    ),
    1: (
        "## 財務分析\n\n"
        "Q1 收入 500，支出 300，利潤率 40%。"
        "Q2 收入成長 20%，但支出也增加 16.7%。"
        "項目 B 狀態異常需要關注。"
    ),
}


@mlflow.trace
def predict_fn(sales_table: str, product_table: str) -> str:
    """模擬 workflow 預測函式。
    
    實際使用時替換為：
        prompt = pm.load_and_format("business_analysis",
                                    sales_table=sales_table,
                                    product_table=product_table)
        kwargs = get_litellm_kwargs(llm_cfg, model_alias="QWEN3")
        response = litellm.completion(**kwargs, messages=[...])
        return response.choices[0].message.content
    """
    # Mock: 根據表格長度選擇回應
    idx = 0 if len(sales_table) > 200 else 1
    return MOCK_RESPONSES[idx]


# 快速測試
test_result = predict_fn(sales_table, product_table)
print(test_result[:200] + "...")

In [ ]:
# === 執行評估 ===
from app.evaluator import run_evaluation
from app.evaluator.scorers import (
    response_not_empty,
    response_length_check,
    contains_keywords,
)

results = run_evaluation(
    predict_fn=predict_fn,
    eval_data=eval_data,
    scorers=[response_not_empty, response_length_check, contains_keywords],
    run_name="multimodel_workflow_eval",
)

print("=== 評估結果 ===")
print(results.tables["eval_results"])

## Step 9: base64 圖片呼叫示範

展示如何使用 `build_multimodal_messages` 建構含圖片的 messages，
並透過 `get_litellm_kwargs` 呼叫 VL 模型。

In [ ]:
from llm_service.factory import build_multimodal_messages

# === 建構含圖片的 messages ===
prompt_text = pm.load_and_format(
    "image_analysis",
    sales_table=sales_table,
)

messages = build_multimodal_messages(
    user_text=prompt_text,
    image_base64=image_base64,
    system_prompt="你是一位視覺分析專家。",
)

print("=== Multimodal Messages 結構 ===")
for msg in messages:
    role = msg["role"]
    if isinstance(msg["content"], list):
        print(f"[{role}] multimodal content:")
        for part in msg["content"]:
            if part["type"] == "text":
                print(f"  - text: {part['text'][:80]}...")
            else:
                url = part["image_url"]["url"]
                print(f"  - image_url: {url[:60]}...")
    else:
        content = msg["content"]
        print(f"[{role}] {content[:80]}...")

# === 呼叫 VL 模型 ===
try:
    kwargs = get_litellm_kwargs(llm_cfg, model_alias="QWEN3VL")
    response = litellm.completion(**kwargs, messages=messages)
    print("\n=== VL 模型分析結果 ===")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"\n[提示] 需要實際 VL 模型服務: {e}")

## 小結

| 步驟 | 說明 | 關鍵 API |
|------|------|----------|
| 配置載入 | 多模型 + 多環境 | `LLMConfig.from_yaml()`, `cfg.resolve("QWEN3")` |
| Prompt 註冊 | MLflow Registry 管理 | `PromptManager.register()`, `mlflow.genai.load_prompt()` |
| Workflow 建構 | LangGraph + WorkflowState | `build_multimodel_chain()`, `create_set_model_node()` |
| 圖片輸入 | base64 multimodal | `build_multimodal_messages()`, `support_images=True` |
| 模型切換 | 動態切換 | `create_set_model_node("QWEN3VL")` |
| 評估 | MLflow GenAI evaluate | `run_evaluation()`, `@scorer` |

### 架構圖

```
llm_config.yaml
  ├── shared_config (auth_urls: DEV/TEST/STG/PROD)
  └── model_configs
        ├── QWEN3  (j1_token, api_endpoints, hyperparams)
        └── QWEN3VL (j1_token, api_endpoints, hyperparams)
              ↓
    LLMConfig.resolve(model_alias, zone)
              ↓
    J1→J2 Token Exchange (自動快取)
              ↓
    ResolvedModelConfig (api_base, api_key, model_name, ...)
              ↓
    get_langchain_llm() / get_litellm_kwargs()
              ↓
    LangGraph WorkflowState (llm_config, model_alias, prompt_template, image_base64)
              ↓
    [preprocess] → [set_model] → [assemble_prompt] → [call_llm] → END
              ↓
    mlflow.genai.evaluate() → Scorers → 評估報告
```